In [ ]:
///- --package .

#### views (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< fixed_views


:= decide_all \* (f : * -> *) {mapped}. (t : policy::thresholds) (observations : f policy::observation) : f i32 =
    mapped (\ o => policy::engagement_code (policy::engages t o)) observations

:= observed (health : i32) (target : policy::reach) : policy::observation =
    { health = health; heal_ready = !+; strike_ready = !+; target = target }

#### one_decision_is_the_live_client_view (test)

In [1]:
///- --test --symbol
///> c

:< policy
:= ~health = 80i32
?= decide_all (evolved_against_cyclops ()) (one (observed health InReach)) ?|
| one code => code |> _assert_eq 1i32

__assert_eq { expected = 1 }

In [ ]:
:= four_reaches (health : i32) : lanes policy::observation =
    :< policy
    := wounded = observed (health - 70) InReach
    lanes (observed health InReach, observed health OutOfReach, wounded, observed health NoTarget)

#### four_lanes_decide_like_four_single_calls (test)

In [2]:
///- --test --symbol
///> c

:= ~health = 80i32
?= decide_all (policy::evolved_against_cyclops ()) (four_reaches health) ?|
| lanes (a, b, c, d) => (a, b, c, d) |> _assert_eq (1i32, 2i32, 0i32, 0i32)

__assert_eq { expected = (1, 2, 0, 0) }

#### a_view_without_mapped_is_rejected (test)

In [3]:
///- --test --symbol --fails "mapped"
///> c

:< policy
decide_all (evolved_against_cyclops ()) (unmapped (observed 80 InReach))

failed to compile as expected ("mapped"): FatalError: File main has a type error somewhere in its path. /home/runner/work/tibia/tibia/src/views.spi:35:46 (main.spi:17:46): ETY-InstanceNotFound: The higher order type instance unmapped does not have the prototype mapped. In the definition: spiral_cells_test. main.spi:19:13: ETY-ValueRestriction: Metavars that are not part of the enclosing function's signature are not allowed. They need to be values. Got: () -> _ i32 In the definition: main.


#### lanes_on_gpu (test)

In [4]:
///- --test --symbol
///> wgsl

:= ~health = 80i32
?= decide_all (policy::evolved_against_cyclops ()) (four_reaches health) ?|
| lanes (a, b, c, d) => ? a = 1i32 && b = 2i32 && c = 0i32 && d = 0i32 ?> 0i32 |: 1i32